# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [1]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.2 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

lecture 01: Using bash command in task 1.1: wc -l data/*.csv > output/file_inventory.txt.

lecture 02: Writing the output to a .csv file (ex. task 2.3 issue_audit.to_csv(AUDIT_PATH, index=False)).

lecture 03: task 4.3 where visit_means = np.mean(sbp_array, axis=0) was used.

axis=0 was chosen to aggregate downwards along the patient rows, collapsing across individual observations to produce a separate mean for each visit column.

lecture 04: Using pandas for dataframe (ex. task 1.2 df_profile = pd.DataFrame(results))

lecture 05: Tasks 3.1 and 3.2 where it was cleaining the table to have the same format.

lecture 06: pd.concat was used (ex. task 2.2 visits_raw = pd.concat([north, south], ignore_index=True))


## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [2]:
%%bash
# paste your command here: wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [3]:
# TODO: write a function, such as profile(name, table), that returns the four values in README

def profile(file_name: str, table: pd.DataFrame) -> dict:
    return {"table": file_name, "rows": len(table), "columns": table.shape[1], "exact_duplicate_rows": int(table.duplicated().sum()),}

# Task 1.2, call it once per name in manifest["files"] with raw[Path(name).stem], and save the results to PROFILE_PATH.
results = []
for name in manifest["files"]:
    table = raw[Path(name).stem]
    results.append(profile(name, table))

df_profile = pd.DataFrame(results)
df_profile.to_csv(PROFILE_PATH, index=False)

## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

The inteneded grain and key for patients_clean is one row per registered patient and patient_id respecitvely. The raw patient table can contain duplicate rows that would violate key uniqueness.

The inteneded grain and key for visits_clean is one row per clinical visit per patient and (patient_id, visit) respecitvely. The violations are vertical stacking issues as some files use wt_kg instead of weight_kg for the column names, identical rows can appear across files, and that multiple evaluations can occur for the same patient and visit window.

The inteneded grain and key for sites_current is one row per clinical trial site and site_code respecitvely. The violations are that the raw csv file contains multiple version records per site, and only record_status == "current" should be used.

The inteneded grain and key for a1c_long is one row patient glycemic laboratory measurement per visit encounter and (patient_id, visit) respecitvely. The violations are that it is stored in wide format with separate columns.

The cardinality I would expect in Task 4.4 are: 1) visits_clean left merge patients_clean on patient_id, with many_to_one cardinality, 2) analysis left merge sites_current on site_code, with many_to_one cardinality, and 3) analysis left merge a1c_long on ["patient_id", "visit"], with one_to_one cardinality.


### 2.2 Stack the two visit files

In [4]:
# TODO: rename wt_kg, add source_file to each visit table, and stack them, north first (README Task 2.2).
visits_raw = None

north = raw["visits_north"].copy(deep=True)
south = raw["visits_south"].copy(deep=True)
south = south.rename(columns={"wt_kg": "weight_kg"})
north["source_file"] = "visits_north.csv"
south["source_file"] = "visits_south.csv"
visits_raw = pd.concat([north, south], ignore_index=True)

### 2.3 Audit the raw tables

In [5]:
# TODO: count the 10 issues in README Task 2.3, in order, as a DataFrame with the columns issue and
# count, and save it to AUDIT_PATH.
issue_audit = None

df_patients = raw["patients"]
df_sites = raw["sites"]
df_visits = visits_raw

gender_stripped = df_patients["gender"].fillna("").astype(str).str.strip()
is_not_empty_gender = gender_stripped != ""
is_not_code = ~df_patients["gender"].astype(str).isin(["F", "M", "X"])
gender_not_code = int((is_not_empty_gender & is_not_code).sum())

empty_gender = int((gender_stripped == "").sum())

birth_year_9999 = int((pd.to_numeric(df_patients["birth_year"], errors="coerce") == 9999).sum())

def normalize_site_code(series: pd.Series) -> pd.Series:
    return series.astype(str).str.upper().str.replace(" ", "", regex=False).str.replace("-", "", regex=False)

site_codes_norm = int((df_patients["site_code"].astype(str) != normalize_site_code(df_patients["site_code"])).sum())

sites_repeated = int(df_sites.duplicated(subset=["site_code"], keep=False).sum())

visits_not_standard = int((~df_visits["visit"].astype(str).isin(["baseline", "week12"])).sum())

sbp_numeric = pd.to_numeric(df_visits["sbp_mmhg"], errors="coerce")
sbp_999 = int((sbp_numeric == 999).sum())

sbp_outside = int(((sbp_numeric != 999) & ((sbp_numeric < 70) | (sbp_numeric > 250))).sum())

date_str = df_visits["visit_date"].astype(str)
matches_format = date_str.str.match(r"^\d{4}-\d{2}-\d{2}$")
parsed_date = pd.to_datetime(df_visits["visit_date"], format="%Y-%m-%d", errors="coerce")
date_failures = int((~(matches_format & parsed_date.notna())).sum())

def map_visit_label(val: pd.Series) -> pd.Series:
    v = val.astype(str).str.strip().str.lower().str.replace("-", "", regex=False).str.replace("_", "", regex=False).str.replace(" ", "", regex=False)
    mapping = {
        "baseline": "baseline",
        "bl": "baseline",
        "base": "baseline",
        "week12": "week12",
        "w12": "week12",
        "wk12": "week12",
        "12weeks": "week12",
        "12w": "week12",}
    return v.map(mapping).fillna(v)

visit_mapped = map_visit_label(df_visits["visit"])
temp_pairs = pd.DataFrame({"patient_id": df_visits["patient_id"], "visit_mapped": visit_mapped,})
repeated_pairs = int(temp_pairs.duplicated(subset=["patient_id", "visit_mapped"], keep=False).sum())

audit_data = [
    {"issue": "patients: gender values that are not a code", "count": gender_not_code},
    {"issue": "patients: empty gender", "count": empty_gender},
    {"issue": "patients: birth_year sentinel 9999", "count": birth_year_9999},
    {"issue": "patients: site codes needing normalization", "count": site_codes_norm},
    {"issue": "sites: rows with a repeated site_code", "count": sites_repeated},
    {"issue": "visits: visit labels that are not baseline or week12", "count": visits_not_standard},
    {"issue": "visits: sbp sentinel 999", "count": sbp_999},
    {"issue": "visits: sbp outside 70 through 250", "count": sbp_outside},
    {"issue": "visits: date parse failures", "count": date_failures},
    {"issue": "visits: rows in repeated patient_id and visit pairs", "count": repeated_pairs},
]

issue_audit = pd.DataFrame(audit_data)
issue_audit.to_csv(AUDIT_PATH, index=False)

assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."

## Task 3: Clean

### 3.1 Clean the patients

In [6]:
# TODO: list the gender spellings with value_counts(), then build patients_clean from
# raw["patients"].copy(deep=True) with steps 1 to 5 in README Task 3.1.
patients_clean = None

patients_clean = raw["patients"].copy(deep=True)
patients_clean = patients_clean.drop_duplicates(keep="first")

gender_map = {
    "f": "F",
    "female": "F",
    "woman": "F",
    "w": "F",
    "m": "M",
    "male": "M",
    "man": "M",
    "x": "X",
    "non-binary": "X",
    "nonbinary": "X",
    "nb": "X",
    "other": "X",}

cleaned_gender = (patients_clean["gender"].astype(str).str.strip().str.lower().map(gender_map))

patients_clean["gender"] = cleaned_gender.astype(pd.CategoricalDtype(categories=["F", "M", "X"]))

birth_year_num = pd.to_numeric(patients_clean["birth_year"], errors="coerce")
patients_clean["birth_year"] = birth_year_num.replace(9999, pd.NA).astype("Int64")

patients_clean["site_code"] = (patients_clean["site_code"].astype(str).str.upper().str.replace("-", "", regex=False).str.replace(" ", "", regex=False))

patients_clean = patients_clean.reset_index(drop=True)

In [7]:
# TODO: show patients_clean["gender"].cat.categories, save the patients summary to
# PATIENTS_SUMMARY_PATH, and save the gender value_counts(dropna=False) with a code column from
# .cat.codes to GENDER_CATEGORIES_PATH, as in README Task 3.1.

print("Gender categories:", patients_clean["gender"].cat.categories)

patients_summary = pd.DataFrame({
    "metric": ["patients","birth_year missing","birth_year min","birth_year max","site_code values","patients at N01","patients at S02","patients at E04",],
    "value": [len(patients_clean), int(patients_clean["birth_year"].isna().sum()), int(patients_clean["birth_year"].min()),
        int(patients_clean["birth_year"].max()), int(patients_clean["site_code"].nunique()), int((patients_clean["site_code"] == "N01").sum()),
        int((patients_clean["site_code"] == "S02").sum()), int((patients_clean["site_code"] == "E04").sum()),],})

patients_summary.to_csv(PATIENTS_SUMMARY_PATH, index=False)

gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()
gender_counts["code"] = gender_counts["gender"].cat.codes
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)

Gender categories: Index(['F', 'M', 'X'], dtype='str')


### 3.2 Clean the visits

In [8]:
# TODO: build visits_clean from visits_raw.copy(deep=True) with steps 1 to 5 in README Task 3.2.
visits_clean = None

visits_clean = visits_raw.copy(deep=True)
visits_clean = visits_clean.drop_duplicates(keep="first")

visit_map = {
    "baseline": "baseline",
    "bl": "baseline",
    "base": "baseline",
    "week12": "week12",
    "wk12": "week12",
    "w12": "week12",
    "week 12": "week12",
    "week-12": "week12",
    "12weeks": "week12",
    "12w": "week12",}

visits_clean["visit"] = (visits_clean["visit"].astype(str).str.strip().str.lower().map(visit_map))

sbp_numeric = pd.to_numeric(visits_clean["sbp_mmhg"], errors="coerce")
visits_clean["sbp_mmhg"] = (sbp_numeric.where((sbp_numeric >= 70) & (sbp_numeric <= 250)).round().astype("Int64"))

visits_clean["weight_kg"] = pd.to_numeric(visits_clean["weight_kg"], errors="coerce")
is_exact_format = visits_clean["visit_date"].astype(str).str.match(r"^\d{4}-\d{2}-\d{2}$")
parsed_dates = pd.to_datetime(visits_clean["visit_date"], format="%Y-%m-%d", errors="coerce")
visits_clean["visit_date"] = parsed_dates.where(is_exact_format)

In [9]:
# TODO: step 6, show the repeated patient_id and visit pairs, then keep the later reading.
repeated_mask = visits_clean.duplicated(subset=["patient_id", "visit"], keep=False)
print("Repeated patient_id and visit pairs (rechecks):")
display(visits_clean[repeated_mask].sort_values(["patient_id", "visit", "visit_time"]))

visits_clean = (visits_clean.sort_values("visit_time").drop_duplicates(subset=["patient_id", "visit"], keep="last"))

Repeated patient_id and visit pairs (rechecks):


,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
26298,P00001,baseline,2026-03-06,14:15,141,52.4,visits_south.csv
26297,P00001,baseline,2026-03-06,14:40,146,52.4,visits_south.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
5,P00004,baseline,2026-02-25,11:15,142,106.2,visits_north.csv
84,P00088,baseline,2026-03-10,09:30,153,68.2,visits_north.csv
...,...,...,...,...,...,...,...
55473,P28232,baseline,2026-01-12,10:10,159,61.0,visits_south.csv
56249,P28953,baseline,2026-01-21,10:30,153,74.8,visits_south.csv
56248,P28953,baseline,2026-01-21,10:45,151,74.8,visits_south.csv
25464,P29170,baseline,2026-03-02,11:20,130,78.0,visits_north.csv


In [10]:
# TODO: step 7, sort, renumber, keep the six columns, then save the visits summary to
# VISITS_SUMMARY_PATH (README Task 3.2).

visits_clean = (visits_clean.sort_values(["patient_id", "visit"]).reset_index(drop=True)[
        ["patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "source_file",]])

print("Cleaned visit counts:")
print(visits_clean["visit"].value_counts(dropna=False))

visits_summary = pd.DataFrame({
    "metric": [
        "visits",
        "baseline visits",
        "week12 visits",
        "visit_date missing",
        "visit_date min",
        "visit_date max",
        "sbp_mmhg missing",
        "sbp_mmhg min",
        "sbp_mmhg max",
        "weight_kg missing",
        "weight_kg mean",
        "visits from visits_north.csv",
        "visits from visits_south.csv",],
    "value": [
        len(visits_clean),
        (visits_clean["visit"] == "baseline").sum(),
        (visits_clean["visit"] == "week12").sum(),
        visits_clean["visit_date"].isna().sum(),
        visits_clean["visit_date"].min(),
        visits_clean["visit_date"].max(),
        visits_clean["sbp_mmhg"].isna().sum(),
        visits_clean["sbp_mmhg"].min(),
        visits_clean["sbp_mmhg"].max(),
        visits_clean["weight_kg"].isna().sum(),
        round(visits_clean["weight_kg"].mean(), 2),
        (visits_clean["source_file"] == "visits_north.csv").sum(),
        (visits_clean["source_file"] == "visits_south.csv").sum(),
    ],})

visits_summary.to_csv(VISITS_SUMMARY_PATH, index=False)

Cleaned visit counts:
visit
baseline    29599
week12      27582
Name: count, dtype: int64


### 3.3 Record the decisions

In [11]:
# TODO: one row per issue in README Task 3.3, with table and issue copied, and your own
# decision and one-sentence reason, then save it to DECISION_LOG_PATH.
decision_table = pd.DataFrame(columns=["table", "issue", "decision", "reason"])
decision_table

decision_data = [
    {"table": "patients",
        "issue": "exact duplicate rows",
        "decision": "dropped identical duplicate rows while only keeping the first occurrence.",
        "reason": "duplicate records reflect redundant logging and artificially inflate the unique patient population count.",
    },
    {"table": "patients",
        "issue": "gender spellings that are not a code",
        "decision": "stripped whitespace, lowercased strings, mapped standard synonyms to categorical codes F, M, and X, and set unlisted or blank values to missing.",
        "reason": "standardizing free-text variants to uniform categorical codes ensures clinical interoperability while avoiding unsubstantiated imputation for ambiguous entries.",
    },
    {"table": "patients",
        "issue": "site codes needing normalization",
        "decision": "converted strings to uppercase and stripped all hyphens and spaces.",
        "reason": "inconsistent formatting and casing represent the same clinic identifiers across sites and must align for accurate relational joins.",
    },
    {"table": "patients",
        "issue": "birth_year sentinel 9999",
        "decision": "replaced the sentinel value 9999 with missing values (pd.NA) and cast the column to Int64.",
        "reason": "the value 9999 is a placeholder and would severely distort age calculations.",
    },
    {"table": "visits",
        "issue": "weight column named wt_kg in visits_south.csv",
        "decision": "renamed wt_kg to weight_kg prior to vertically concatenating the visit tables.",
        "reason": "coordinating column names ensures attributes align into a single unified column without introducing empty split fields upon stacking.",
    },
    {"table": "visits",
        "issue": "sbp sentinel 999 and readings outside 70 through 250",
        "decision": "converted values to numeric, retained readings from 70 through 250, and set the 999 sentinel and out-of-range values to missing.",
        "reason": "readings outside physiological limits and placeholder codes can distort clinical summary statistics and downstream blood pressure modeling.",
    },
    {"table": "visits",
        "issue": "impossible and loosely written visit dates",
        "decision": "enforced strict YYYY-MM-DD formatting against a real calendar date, converting non-matching or invalid entries to missing.",
        "reason": "accepting only valid calendar dates prevents misclassification and incorrect interval derivations across visits.",
    },
    {"table": "visits",
        "issue": "repeated patient_id and visit pairs",
        "decision": "sorted records chronologically by visit_time and retained the last reading for each patient and visit combination.",
        "reason": "repeated pairs represent clinical rechecks during the same encounter, making the latest measurement the intended final reading.",
    },
    {"table": "sites",
        "issue": "repeated site_code rows",
        "decision": "removed duplicate site entries to maintain a single unique record per site code.",
        "reason": "primary keys in dimension tables must be unique.",
    },]

decision_table = pd.DataFrame(decision_data)
decision_table.to_csv(DECISION_LOG_PATH, index=False)

## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [12]:
# TODO: melt raw["a1c_wide"], map the visit labels, drop the rows with no result,
# then save the per-visit summary to A1C_SUMMARY_PATH (README Task 4.1).
a1c_long = None

a1c_long = raw["a1c_wide"].melt(id_vars=["patient_id"], value_vars=["a1c_baseline", "a1c_week12"], var_name="visit", value_name="a1c_pct",)

a1c_long["visit"] = a1c_long["visit"].map({"a1c_baseline": "baseline", "a1c_week12": "week12",})

assert a1c_long["a1c_pct"].dtype == "float64", "a1c_pct is not float64"
a1c_long = a1c_long.dropna(subset=["a1c_pct"]).reset_index(drop=True)

a1c_summary = pd.DataFrame([
    {"visit": "baseline",
        "results": int((a1c_long["visit"] == "baseline").sum()),
        "a1c_pct_mean": round(a1c_long.loc[a1c_long["visit"] == "baseline", "a1c_pct"].mean(), 2),
    },
    {"visit": "week12",
        "results": int((a1c_long["visit"] == "week12").sum()),
        "a1c_pct_mean": round(a1c_long.loc[a1c_long["visit"] == "week12", "a1c_pct"].mean(), 2),
    },])

a1c_summary.to_csv(A1C_SUMMARY_PATH, index=False)

### 4.2 Check the keys before joining

In [13]:
# TODO: steps 1 and 2, catch the MergeError, show the repeated site rows, and build sites_current.
sites = raw["sites"]
sites_current = None

try:
    patients_clean.merge(sites, on="site_code", how="left", validate="many_to_one")
except pd.errors.MergeError as e:
    print(f"Caught expected MergeError: {e}")

repeated_sites = sites[sites.duplicated(subset=["site_code"], keep=False)]
print("\nRepeated site rows:")
display(repeated_sites)

sites_current = (sites[sites["record_status"] == "current"][["site_code", "site_name"]].reset_index(drop=True))
print("\nsites_current:")
display(sites_current)

Caught expected MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 site_code
      N01
      S02 ...

Repeated site rows:


,site_code,site_name,record_status
0,N01,North Clinic,current
1,N01,North Street Clinic,retired
2,S02,Southgate Clinic,retired
3,S02,South Valley Clinic,current



sites_current:


,site_code,site_name
0,N01,North Clinic
1,S02,South Valley Clinic
2,W03,West Clinic


In [14]:
# TODO: step 3, outer merge visits_clean and a1c_long with indicator=True and show the _merge counts,
# then save the seven counts in README Task 4.2 to MERGE_AUDIT_PATH.

visits_a1c_outer = visits_clean.merge(a1c_long, on=["patient_id", "visit"], how="outer",indicator=True)

print("_merge counts for visits_clean and a1c_long:")
print(visits_a1c_outer["_merge"].value_counts())

count_both = int((visits_a1c_outer["_merge"] == "both").sum())
count_left_only = int((visits_a1c_outer["_merge"] == "left_only").sum())
count_right_only = int((visits_a1c_outer["_merge"] == "right_only").sum())

count_visits_not_in_reg = int((~visits_clean["patient_id"].isin(patients_clean["patient_id"])).sum())
count_reg_not_in_visits = int((~patients_clean["patient_id"].isin(visits_clean["patient_id"])).sum())

count_reg_site_not_in_sites = int((~patients_clean["site_code"].isin(sites["site_code"])).sum())
count_site_removed_rule = int(len(sites) - len(sites_current))

merge_audit = pd.DataFrame([
    {"check": "visits with an A1c result", "count": count_both},
    {"check": "visits without an A1c result", "count": count_left_only},
    {"check": "A1c results without a visit", "count": count_right_only},
    {"check": "visits whose patient is not in the registry", "count": count_visits_not_in_reg},
    {"check": "registry patients with no visit", "count": count_reg_not_in_visits},
    {"check": "registry patients whose site is not in the site list", "count": count_reg_site_not_in_sites},
    {"check": "site list rows removed by the current-record rule", "count": count_site_removed_rule},
])

merge_audit.to_csv(MERGE_AUDIT_PATH, index=False)
display(merge_audit)

_merge counts for visits_clean and a1c_long:
_merge
both          54006
left_only      3175
right_only     2754
Name: count, dtype: int64


,check,count
0,visits with an A1c result,54006
1,visits without an A1c result,3175
2,A1c results without a visit,2754
3,visits whose patient is not in the registry,79
4,registry patients with no visit,441
5,registry patients whose site is not in the sit...,555
6,site list rows removed by the current-record rule,2


### 4.3 Pivot blood pressure to one row per patient

In [15]:
# TODO: pivot visits_clean, reset the index, clear columns.name, add sbp_change, then save the
# SAMPLE_IDS rows to SBP_WIDE_SAMPLE_PATH (README Task 4.3).
sbp_wide = None

sbp_wide = visits_clean.pivot(index="patient_id", columns="visit", values="sbp_mmhg").reset_index()
sbp_wide.columns.name = None
sbp_wide["sbp_change"] = sbp_wide["week12"] - sbp_wide["baseline"]
sbp_sample = sbp_wide[sbp_wide["patient_id"].isin(SAMPLE_IDS)]
sbp_sample.to_csv(SBP_WIDE_SAMPLE_PATH, index=False)
complete_pairs = sbp_wide[["baseline", "week12"]].dropna()

#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [16]:
# TODO: select complete pairs, convert to a NumPy array, and display its shape and two per-visit means.

complete_pairs = sbp_wide[["baseline", "week12"]].dropna()
sbp_array = np.array(complete_pairs, dtype=float)

print(f"Shape of complete_pairs array: {sbp_array.shape}")
visit_means = np.mean(sbp_array, axis=0)
print(f"Baseline mean: {visit_means[0]:.2f} mmHg")
print(f"Week 12 mean:  {visit_means[1]:.2f} mmHg")

Shape of complete_pairs array: (27149, 2)
Baseline mean: 143.01 mmHg
Week 12 mean:  137.15 mmHg


### 4.4 Join the analysis table

In [17]:
# TODO: three left merges onto visits_clean, each with validate=, then the columns and dtypes
# in README Task 4.4. A MergeError means an earlier step left a problem: see HINTS.md.
analysis = None

analysis = visits_clean.merge(patients_clean, on="patient_id", how="left", validate="many_to_one",)
analysis = analysis.merge(sites_current, on="site_code", how="left", validate="many_to_one",)
analysis = analysis.merge(a1c_long, on=["patient_id", "visit"], how="left", validate="one_to_one",)

column_order = ["patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "a1c_pct", "gender", "birth_year", "site_code", "site_name", "source_file",]

analysis = analysis[column_order].astype({
    "patient_id": "string",
    "visit": "string",
    "visit_date": "datetime64[us]",
    "sbp_mmhg": "Int64",
    "weight_kg": "float64",
    "a1c_pct": "float64",
    "gender": "string",
    "birth_year": "Int64",
    "site_code": "string",
    "site_name": "string",
    "source_file": "string",
})

## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [18]:
# TODO: collect named True/False checks in a Series (README Task 5.1), display it, and stop with
# assert checks.all(), checks[~checks] before the analysis extract is saved.
# A failed check means an earlier step left a problem: print checks[~checks] and see HINTS.md.

checks = pd.Series({
    "analysis has as many rows as visits_clean": (len(analysis) == len(visits_clean)),
    "no patient_id and visit pair repeats in analysis": (~analysis.duplicated(subset=["patient_id", "visit"]).any()),
    "every sbp_mmhg that is present in analysis is from 70 through 250": (analysis["sbp_mmhg"].dropna().between(70, 250).all()),
    "every site_name that is present is a current site name": (analysis["site_name"].dropna().isin(sites_current["site_name"]).all()),
    "sbp_wide['patient_id'] is unique": (sbp_wide["patient_id"].is_unique),
    "every raw table still equals its snapshot": (all(raw[name].equals(raw_snapshot[name]) for name in raw)),})

display(checks)
assert checks.all(), checks[~checks]

analysis has as many rows as visits_clean                            True
no patient_id and visit pair repeats in analysis                     True
every sbp_mmhg that is present in analysis is from 70 through 250    True
every site_name that is present is a current site name               True
sbp_wide['patient_id'] is unique                                     True
every raw table still equals its snapshot                            True
dtype: bool

### 5.2 Save the analysis extract

In [19]:
# TODO: keep the analysis rows whose patient_id is in SAMPLE_IDS, renumber them, and save them to
# ANALYSIS_SAMPLE_PATH with index=False (README Task 5.2).

analysis_sample = (analysis[analysis["patient_id"].isin(SAMPLE_IDS)].reset_index(drop=True))

expected_rows = visits_clean["patient_id"].isin(SAMPLE_IDS).sum()
assert len(analysis_sample) == expected_rows, (f"Expected {expected_rows} rows, but got {len(analysis_sample)}")

analysis_sample.to_csv(ANALYSIS_SAMPLE_PATH, index=False)

### 5.3 Read the file back

In [20]:
# TODO: read ANALYSIS_SAMPLE_PATH back with its dtypes and parse_dates=["visit_date"], and print
# whether it equals analysis_sample.

round_trip = pd.read_csv(
    ANALYSIS_SAMPLE_PATH,
    dtype={
        "patient_id": "string",
        "visit": "string",
        "sbp_mmhg": "Int64",
        "weight_kg": "float64",
        "a1c_pct": "float64",
        "gender": "string",
        "birth_year": "Int64",
        "site_code": "string",
        "site_name": "string",
        "source_file": "string",
    },
    parse_dates=["visit_date"],
)

round_trip["visit_date"] = round_trip["visit_date"].astype("datetime64[us]")
print(round_trip.equals(analysis_sample))

True
